In [102]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
from scipy.stats import wasserstein_distance

In [110]:
data = pd.read_excel('../dataframe/BD_Atlas_1991_2024_v1.0_2025.04.14_Consolidado.xlsx')
data.shape

(71929, 70)

In [112]:
data =data[['Nome_Municipio', 'Sigla_UF', 'descricao_tipologia', 'grupo_de_desastre','DH_MORTOS', 'DH_FERIDOS', 'DH_DESABRIGADOS','DA_Polui/cont do solo', 'DA_Polui/cont do ar', 'PEPL_total_publico', 'PEPR_total_privado']]
data


,Nome_Municipio,Sigla_UF,descricao_tipologia,grupo_de_desastre,DH_MORTOS,DH_FERIDOS,DH_DESABRIGADOS,DA_Polui/cont do solo,DA_Polui/cont do ar,PEPL_total_publico,PEPR_total_privado
0,Salto Veloso,SC,Estiagem e Seca,Climatológico,0,0,0,NaN,NaN,0.0,0.0
1,Nova Palma,RS,Estiagem e Seca,Climatológico,0,0,0,NaN,NaN,0.0,0.0
2,Caseiros,RS,Estiagem e Seca,Climatológico,0,0,0,NaN,NaN,0.0,0.0
3,Jaborá,SC,Estiagem e Seca,Climatológico,0,0,0,NaN,NaN,0.0,0.0
4,Iporã do Oeste,SC,Estiagem e Seca,Climatológico,0,0,0,NaN,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
71924,Caconde,SP,Inundações,Hidrológico,0,0,0,NaN,NaN,0.0,0.0
71925,Tanabi,SP,Chuvas Intensas,Hidrológico,0,0,0,NaN,NaN,0.0,0.0
71926,Rio Rufino,SC,Granizo,Meteorológico,0,0,0,NaN,NaN,0.0,0.0
71927,Luminárias,MG,Chuvas Intensas,Hidrológico,0,0,0,NaN,NaN,0.0,0.0


# 🎲 Limpeza de dados

In [113]:
import pandas as pd
import numpy as np
import re

def sanitize_column_names(df):
    """
    Padroniza nomes das colunas: minúsculas, sem espaços e caracteres especiais.
    """
    df.columns = [
        re.sub(r'\W+', '_', col.strip().lower()) for col in df.columns
    ]
    return df

def preprocess_disaster_data(df):
    """
    Sanitiza e pré-processa o DataFrame de dados de desastres para uso em modelo de simulação.

    Etapas:
    - Renomeia colunas com nomes limpos
    - Binariza colunas de poluição
    - Converte colunas categóricas para tipo 'category'
    - Preenche valores ausentes com 0 ou categoria 'desconhecido'
    - Garante que colunas numéricas fiquem com valores válidos
    """

    # Renomear colunas
    df = sanitize_column_names(df)

    # Identificar colunas categóricas
    categorical_cols = df.select_dtypes(include='object').columns.tolist()

    # Converter para categoria e preencher nulos com 'desconhecido'
    for col in categorical_cols:
        df[col] = df[col].fillna('desconhecido').astype('category')

    # Preencher valores nulos numéricos com 0
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df[numeric_cols] = df[numeric_cols].fillna(0)

    return df

In [114]:
data = preprocess_disaster_data(data)
data

,nome_municipio,sigla_uf,descricao_tipologia,grupo_de_desastre,dh_mortos,dh_feridos,dh_desabrigados,da_polui_cont_do_solo,da_polui_cont_do_ar,pepl_total_publico,pepr_total_privado
0,Salto Veloso,SC,Estiagem e Seca,Climatológico,0,0,0,desconhecido,desconhecido,0.0,0.0
1,Nova Palma,RS,Estiagem e Seca,Climatológico,0,0,0,desconhecido,desconhecido,0.0,0.0
2,Caseiros,RS,Estiagem e Seca,Climatológico,0,0,0,desconhecido,desconhecido,0.0,0.0
3,Jaborá,SC,Estiagem e Seca,Climatológico,0,0,0,desconhecido,desconhecido,0.0,0.0
4,Iporã do Oeste,SC,Estiagem e Seca,Climatológico,0,0,0,desconhecido,desconhecido,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
71924,Caconde,SP,Inundações,Hidrológico,0,0,0,desconhecido,desconhecido,0.0,0.0
71925,Tanabi,SP,Chuvas Intensas,Hidrológico,0,0,0,desconhecido,desconhecido,0.0,0.0
71926,Rio Rufino,SC,Granizo,Meteorológico,0,0,0,desconhecido,desconhecido,0.0,0.0
71927,Luminárias,MG,Chuvas Intensas,Hidrológico,0,0,0,desconhecido,desconhecido,0.0,0.0


# Monte carlo.

In [119]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Supondo que 'data' já está carregado em df
df = data.copy()

# Criar dicionário para armazenar encoders
label_encoders = {}

# Features para Label Encoding
categorical_features = ['descricao_tipologia', 'grupo_de_desastre']

# Aplicar LabelEncoder em cada feature categórica
for col in categorical_features:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

# Preparar df para simulação mantendo 'nome_municipio' e 'sigla_uf' originais
df_proc = df[[
    'nome_municipio',
    'sigla_uf',
    'descricao_tipologia_encoded',
    'grupo_de_desastre_encoded',
    'dh_mortos',
    'dh_feridos',
    'dh_desabrigados',
    'da_polui_cont_do_solo',
    'da_polui_cont_do_ar',
    'pepl_total_publico',
    'pepr_total_privado',
]]

# Simulação Monte Carlo: 1000 amostras bootstrap
num_simulacoes = 1000
simulacoes = df_proc.sample(n=num_simulacoes, replace=True, random_state=42).reset_index(drop=True)

# Reverter Label Encoding para as colunas categóricas codificadas
for col in categorical_features:
    simulacoes[col] = label_encoders[col].inverse_transform(simulacoes[col + '_encoded'])

# Remover colunas codificadas para saída final
simulacoes = simulacoes.drop(columns=[col + '_encoded' for col in categorical_features])

# Salvar arquivo CSV final
simulacoes.to_csv("simulacoes_monte_carlo_final.csv", index=False)


In [120]:
simulacoes

,nome_municipio,sigla_uf,dh_mortos,dh_feridos,dh_desabrigados,da_polui_cont_do_solo,da_polui_cont_do_ar,pepl_total_publico,pepr_total_privado,descricao_tipologia,grupo_de_desastre
0,Capão Alto,SC,0,0,0,desconhecido,desconhecido,0.00,5692000.0,Estiagem e Seca,Climatológico
1,Cunha Porã,SC,0,0,0,desconhecido,desconhecido,0.00,0.0,Enxurradas,Hidrológico
2,Camapuã,MS,0,0,0,desconhecido,desconhecido,0.00,0.0,Estiagem e Seca,Climatológico
3,São João da Lagoa,MG,0,0,0,desconhecido,desconhecido,0.00,0.0,Estiagem e Seca,Climatológico
4,Limoeiro do Norte,CE,0,0,0,desconhecido,desconhecido,0.00,0.0,Vendavais e Ciclones,Meteorológico
...,...,...,...,...,...,...,...,...,...,...,...
995,Caputira,MG,0,0,0,desconhecido,desconhecido,0.00,0.0,Granizo,Meteorológico
996,Antônio Olinto,PR,0,0,0,desconhecido,desconhecido,0.00,0.0,Vendavais e Ciclones,Meteorológico
997,São Francisco,MG,0,0,0,desconhecido,desconhecido,198999.99,6747986.0,Estiagem e Seca,Climatológico
998,Canindé de São Francisco,SE,0,0,0,desconhecido,desconhecido,0.00,0.0,Estiagem e Seca,Climatológico


# Resultados.

In [123]:
import pandas as pd
import numpy as np
from scipy.stats import skew

df_original = data
df_sim = simulacoes

numericas = ['dh_mortos', 'dh_feridos', 'dh_desabrigados', 'pepl_total_publico', 'pepr_total_privado']

sample_size = 10
resultados = []

for idx, sim_row in df_sim.iterrows():
    municipio = sim_row['nome_municipio']

    # Filtra dados originais do mesmo município
    df_mun = df_original[df_original['nome_municipio'] == municipio]

    # Se não tiver dados suficientes, pula ou pega tudo
    if len(df_mun) == 0:
        continue
    elif len(df_mun) < sample_size:
        amostra = df_mun
    else:
        amostra = df_mun.sample(sample_size, random_state=42)

    # Para cada variável numérica, calcular diferença média, diferença mediana, e skewness original x simulado
    dif_media = {}
    dif_mediana = {}
    skew_orig = {}
    skew_sim = {}

    for var in numericas:
        media_original = amostra[var].mean()
        mediana_original = amostra[var].median()
        skew_original = skew(amostra[var])

        valor_simulado = sim_row[var]

        dif_media[var] = abs(media_original - valor_simulado)
        dif_mediana[var] = abs(mediana_original - valor_simulado)
        skew_orig[var] = skew_original
        skew_sim[var] = 0  # só uma linha simulada, skew não aplicável diretamente

    resultados.append({
        'idx_sim': idx,
        'nome_municipio': municipio,
        'dif_media': dif_media,
        'dif_mediana': dif_mediana,
        'skew_orig': skew_orig,
        'skew_sim': skew_sim
    })

# Converter resultados para DataFrame legível
import json

def expand_dict_col(df, col_name, prefix):
    # Expande coluna com dict para colunas separadas
    return pd.concat([
        df.drop(columns=[col_name]),
        df[col_name].apply(pd.Series).add_prefix(prefix)
    ], axis=1)

df_resultados = pd.DataFrame(resultados)
df_resultados = expand_dict_col(df_resultados, 'dif_media', 'dif_media_')
df_resultados = expand_dict_col(df_resultados, 'dif_mediana', 'dif_mediana_')
df_resultados = expand_dict_col(df_resultados, 'skew_orig', 'skew_orig_')
df_resultados = expand_dict_col(df_resultados, 'skew_sim', 'skew_sim_')

# Mostrar métricas resumidas
print("Média das diferenças absolutas das médias por variável:")
print(df_resultados.filter(like='dif_media_').mean())

print("\nMédia das diferenças absolutas das medianas por variável:")
print(df_resultados.filter(like='dif_mediana_').mean())

print("\nMédia do skewness original por variável:")
print(df_resultados.filter(like='skew_orig_').mean())

Média das diferenças absolutas das médias por variável:
dif_media_dh_mortos             9.715357e-02
dif_media_dh_feridos            2.205277e+00
dif_media_dh_desabrigados       5.822499e+01
dif_media_pepl_total_publico    7.333066e+05
dif_media_pepr_total_privado    6.207802e+06
dtype: float64

Média das diferenças absolutas das medianas por variável:
dif_mediana_dh_mortos             6.600000e-02
dif_mediana_dh_feridos            1.132000e+00
dif_mediana_dh_desabrigados       3.240850e+01
dif_mediana_pepl_total_publico    4.927499e+05
dif_mediana_pepr_total_privado    3.749047e+06
dtype: float64

Média do skewness original por variável:
skew_orig_dh_mortos             2.392404
skew_orig_dh_feridos            2.315319
skew_orig_dh_desabrigados       2.153484
skew_orig_pepl_total_publico    1.822221
skew_orig_pepr_total_privado    1.750724
dtype: float64


In [87]:
numericas = ['dh_mortos', 'dh_feridos', 'dh_desabrigados', 'PEPL_total_publico', 'PEPR_total_privado']

municipios = df_sim['nome_municipio'].unique()

resultados = []

for municipio in municipios:
    original_vals = df_original[df_original['nome_municipio'] == municipio]
    sim_vals = df_sim[df_sim['nome_municipio'] == municipio]

    if len(original_vals) < 5 or len(sim_vals) < 5:
        # Ignorar municípios com amostras muito pequenas
        continue

    for var in numericas:
        data_orig = original_vals[var]
        data_sim = sim_vals[var]

        # KS Test
        ks_stat, ks_pvalue = ks_2samp(data_orig, data_sim)

        # Wasserstein Distance
        wass_dist = wasserstein_distance(data_orig, data_sim)

        resultados.append({
            'nome_municipio': municipio,
            'variavel': var,
            'ks_stat': ks_stat,
            'ks_pvalue': ks_pvalue,
            'wasserstein_dist': wass_dist
        })

df_resultados = pd.DataFrame(resultados)

# Mostrar resultados resumidos por variável (média dos municípios)
print("Resumo das métricas por variável:")
print(df_resultados.groupby('variavel')[['ks_stat','ks_pvalue','wasserstein_dist']].mean())

Resumo das métricas por variável:
                  ks_stat  ks_pvalue  wasserstein_dist
variavel                                              
dh_desabrigados  0.004464        1.0         12.111607
dh_feridos       0.017857        1.0          2.254464
dh_mortos        0.026786        1.0          0.062500


In [124]:
import matplotlib.pyplot as plt
import seaborn as sns

def analisar_e_plotar(df_comparacao):
    col_diff_categoricos = [col for col in df_comparacao.columns if col.startswith('diff_sigla_uf') or
                            col.startswith('diff_descricao_tipologia') or
                            col.startswith('diff_grupo_de_desastre') or
                            col.startswith('diff_da_polui_cont_do_solo') or
                            col.startswith('diff_da_polui_cont_do_ar')]
    col_diff_numericos = [col for col in df_comparacao.columns if col.startswith('diff_dh_')]

    # Colunas médias e geral
    medias = ['diff_categoricos_media', 'diff_numericos_media', 'diff_geral']

    print("=== Métricas de diferenças categóricas ===")
    print(df_comparacao[[col for col in col_diff_categoricos]].describe().T[['mean', '50%', 'std']])
    print("\n=== Métricas de diferenças numéricas ===")
    print(df_comparacao[[col for col in col_diff_numericos]].describe().T[['mean', '50%', 'std']])
    print("\n=== Métricas agregadas ===")
    print(df_comparacao[medias].describe().T[['mean', '50%', 'std']])

    # Plotar histogramas das médias agregadas
    plt.figure(figsize=(15,4))
    for i, col in enumerate(medias):
        plt.subplot(1, 3, i+1)
        sns.histplot(df_comparacao[col].dropna(), bins=30, kde=True)
        plt.title(f'Distribuição de {col}')
        plt.xlabel('Diferença')
    plt.tight_layout()
    plt.show()

    # Plotar boxplots por campo categórico e numérico
    plt.figure(figsize=(14,6))
    plt.subplot(1,2,1)
    sns.boxplot(data=df_comparacao[col_diff_categoricos])
    plt.title('Diferenças nas Features Categóricas')
    plt.xticks(rotation=45)

    plt.subplot(1,2,2)
    sns.boxplot(data=df_comparacao[col_diff_numericos])
    plt.title('Diferenças nas Features Numéricas')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()


,nome_municipio,sigla_uf,dh_mortos,dh_feridos,dh_desabrigados,da_polui_cont_do_solo,da_polui_cont_do_ar,descricao_tipologia,grupo_de_desastre,diff_sigla_uf,diff_descricao_tipologia,diff_grupo_de_desastre,diff_da_polui_cont_do_solo,diff_da_polui_cont_do_ar,diff_dh_mortos,diff_dh_feridos,diff_dh_desabrigados
0,Capão Alto,SC,0,0,0,desconhecido,desconhecido,Estiagem e Seca,Climatológico,0,1,1,0,0,0.0,0.8,0.000000
1,Cunha Porã,SC,0,0,0,desconhecido,desconhecido,Enxurradas,Hidrológico,0,1,1,0,0,0.0,0.0,0.000000
2,Camapuã,MS,0,0,0,desconhecido,desconhecido,Estiagem e Seca,Climatológico,0,0,0,0,0,0.0,0.0,0.000000
3,São João da Lagoa,MG,0,0,0,desconhecido,desconhecido,Estiagem e Seca,Climatológico,0,0,0,0,0,0.0,0.0,0.000000
4,Limoeiro do Norte,CE,0,0,0,desconhecido,desconhecido,Vendavais e Ciclones,Meteorológico,0,1,1,0,0,0.0,0.0,0.998088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Caputira,MG,0,0,0,desconhecido,desconhecido,Granizo,Meteorológico,0,1,1,0,0,0.0,0.0,0.000000
996,Antônio Olinto,PR,0,0,0,desconhecido,desconhecido,Vendavais e Ciclones,Meteorológico,0,0,0,0,0,0.0,0.0,0.000000
997,São Francisco,MG,0,0,0,desconhecido,desconhecido,Estiagem e Seca,Climatológico,0,0,0,0,0,0.0,0.0,0.000000
998,Canindé de São Francisco,SE,0,0,0,desconhecido,desconhecido,Estiagem e Seca,Climatológico,0,0,0,0,0,0.0,0.0,0.000000


# Michael


In [130]:
simulacoes = pd.read_csv('../dataframe/mil_simulacoes_desastres.csv', sep = ";")
simulacoes

,nome_municipio,sigla_uf,dh_mortos,dh_feridos,dh_desabrigados,da_polui_cont_do_solo,da_polui_cont_do_ar,descricao_tipologia,grupo_de_desastre
0,Catalão,SP,0,1,11,0,0,Incêndios florestais,Incêndios Florestais
1,Manaus,AP,0,2,10,0,0,Inundações graduais,Desastres Biológicos (ex: epidemias relacionad...
2,Curitiba,PR,0,0,5,0,0,Deslizamentos de terra,Hidrológico
3,Anápolis,AC,0,3,8,0,0,Enxurradas,Hidrológico
4,Curitiba,AC,0,0,8,0,0,Secas e estiagens,Geológico
...,...,...,...,...,...,...,...,...,...
995,Catalão,RJ,1,3,8,0,0,Inundações graduais,Meteorológico
996,Recife,MT,0,2,7,0,0,Enxurradas,Desastres Tecnológicos (simulando contaminação...
997,Aparecida de Goiânia,RO,1,1,8,0,0,Secas e estiagens,Hidrológico
998,Anápolis,RJ,0,0,11,0,0,Deslizamentos de terra,Climatológico


In [132]:
import pandas as pd
import numpy as np
from scipy.stats import skew

df_original = data
df_sim = simulacoes

numericas = ['dh_mortos', 'dh_feridos', 'dh_desabrigados']

sample_size = 10
resultados = []

for idx, sim_row in df_sim.iterrows():
    municipio = sim_row['nome_municipio']

    # Filtra dados originais do mesmo município
    df_mun = df_original[df_original['nome_municipio'] == municipio]

    # Se não tiver dados suficientes, pula ou pega tudo
    if len(df_mun) == 0:
        continue
    elif len(df_mun) < sample_size:
        amostra = df_mun
    else:
        amostra = df_mun.sample(sample_size, random_state=42)

    # Para cada variável numérica, calcular diferença média, diferença mediana, e skewness original x simulado
    dif_media = {}
    dif_mediana = {}
    skew_orig = {}
    skew_sim = {}

    for var in numericas:
        media_original = amostra[var].mean()
        mediana_original = amostra[var].median()
        skew_original = skew(amostra[var])

        valor_simulado = sim_row[var]

        dif_media[var] = abs(media_original - valor_simulado)
        dif_mediana[var] = abs(mediana_original - valor_simulado)
        skew_orig[var] = skew_original
        skew_sim[var] = 0  # só uma linha simulada, skew não aplicável diretamente

    resultados.append({
        'idx_sim': idx,
        'nome_municipio': municipio,
        'dif_media': dif_media,
        'dif_mediana': dif_mediana,
        'skew_orig': skew_orig,
        'skew_sim': skew_sim
    })

# Converter resultados para DataFrame legível
import json

def expand_dict_col(df, col_name, prefix):
    # Expande coluna com dict para colunas separadas
    return pd.concat([
        df.drop(columns=[col_name]),
        df[col_name].apply(pd.Series).add_prefix(prefix)
    ], axis=1)

df_resultados = pd.DataFrame(resultados)
df_resultados = expand_dict_col(df_resultados, 'dif_media', 'dif_media_')
df_resultados = expand_dict_col(df_resultados, 'dif_mediana', 'dif_mediana_')
df_resultados = expand_dict_col(df_resultados, 'skew_orig', 'skew_orig_')
df_resultados = expand_dict_col(df_resultados, 'skew_sim', 'skew_sim_')

# Mostrar métricas resumidas
print("Média das diferenças absolutas das médias por variável:")
print(df_resultados.filter(like='dif_media_').mean())

print("\nMédia das diferenças absolutas das medianas por variável:")
print(df_resultados.filter(like='dif_mediana_').mean())

print("\nMédia do skewness original por variável:")
print(df_resultados.filter(like='skew_orig_').mean())

Média das diferenças absolutas das médias por variável:
dif_media_dh_mortos            0.706550
dif_media_dh_feridos           8.702067
dif_media_dh_desabrigados    291.863400
dtype: float64

Média das diferenças absolutas das medianas por variável:
dif_mediana_dh_mortos           0.2460
dif_mediana_dh_feridos          1.4905
dif_mediana_dh_desabrigados    10.4260
dtype: float64

Média do skewness original por variável:
skew_orig_dh_mortos          1.810643
skew_orig_dh_feridos         1.865700
skew_orig_dh_desabrigados    2.027876
dtype: float64
